# FER2013 Facial Expression Recognition
## Assignment 4 — Machine Learning 2026

### Setup order:
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Run **Section 1** (install + clone repo)
3. Run **Section 2** (Kaggle + download data)
4. Run **Section 3** (Wandb login)
5. Run **Section 4** (train all experiments)


## Section 1 — Install dependencies & clone repo

In [ ]:
# Install all required packages
!pip install -q torch torchvision wandb kaggle tqdm seaborn scikit-learn

# Clone YOUR GitHub repo (replace with your actual repo URL after creating it)
# REPO_URL = "https://github.com/YOUR_USERNAME/fer2013-emotion"
# !git clone $REPO_URL
# %cd fer2013-emotion

# For now, just create the folder structure directly
import os
os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
print('✅ Setup complete')

## Section 2 — Download FER2013 from Kaggle

In [ ]:
# Option A: Upload kaggle.json manually
# Go to kaggle.com → Account → Create New API Token → download kaggle.json
from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()  # select kaggle.json when prompted

import os, json
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(json.loads(list(uploaded.values())[0]), f)

os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ Kaggle credentials saved')

In [ ]:
# Download and unzip the FER2013 dataset
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge -p data/
!unzip -q data/challenges-in-representation-learning-facial-expression-recognition-challenge.zip -d data/
!ls data/
print('✅ Dataset downloaded')

## Section 3 — Login to Weights & Biases

In [ ]:
import wandb
# This will prompt you to paste your API key from wandb.ai/authorize
wandb.login()

## Section 4 — Copy source files (if not using git clone)

In [ ]:
# If you cloned your repo, skip this cell.
# Otherwise, upload the src/ files manually:
from google.colab import files
print('Upload all files from the src/ folder (data.py, models.py, sanity_checks.py, utils.py, train.py):')
uploaded = files.upload()

import os
os.makedirs('src', exist_ok=True)
for fname, content in uploaded.items():
    with open(f'src/{fname}', 'wb') as f:
        f.write(content)
    print(f'  saved src/{fname}')
print('✅ Source files ready')

## Section 5 — Quick sanity check before training

In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
from models import get_model, count_parameters
from sanity_checks import run_all_sanity_checks

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Test all three architectures
for arch in ['shallow', 'medium', 'resnet']:
    model = get_model(arch).to(device)
    print(f'\n{arch}: {count_parameters(model):,} parameters')
    run_all_sanity_checks(model, device, arch, log_to_wandb=False)

## Section 6 — Run Architecture 1: ShallowCNN

In [ ]:
WANDB_PROJECT = 'fer2013-emotion'   # change to your project name
WANDB_ENTITY  = ''                  # your wandb username (or leave empty)
CSV_PATH      = 'data/train.csv'

entity_flag = f'--wandb_entity {WANDB_ENTITY}' if WANDB_ENTITY else ''
base = f'python src/train.py --csv {CSV_PATH} --wandb_project {WANDB_PROJECT} {entity_flag}'

# Best config
!{base} --arch shallow --epochs 40 --lr 1e-3 --dropout 0.3 --batch_size 64 --run_name shallow_best

In [ ]:
# Intentional OVERFIT (no dropout, high LR) — required by rubric
!{base} --arch shallow --epochs 40 --lr 5e-3 --dropout 0.0 --weight_decay 0.0 --run_name shallow_OVERFIT

In [ ]:
# Intentional UNDERFIT (too much dropout, tiny LR) — required by rubric
!{base} --arch shallow --epochs 40 --lr 1e-4 --dropout 0.7 --run_name shallow_UNDERFIT

## Section 7 — Run Architecture 2: MediumCNN

In [ ]:
# Best config
!{base} --arch medium --epochs 60 --lr 3e-4 --dropout 0.25 --weight_decay 1e-4 --run_name medium_best

In [ ]:
# High LR variant (overfit / unstable)
!{base} --arch medium --epochs 60 --lr 1e-3 --dropout 0.25 --weight_decay 0.0 --run_name medium_highlr

In [ ]:
# More regularisation (underfit territory)
!{base} --arch medium --epochs 60 --lr 1e-4 --dropout 0.5 --weight_decay 1e-3 --run_name medium_heavy_reg

In [ ]:
# Large batch (often underfits with same LR)
!{base} --arch medium --epochs 60 --lr 3e-4 --dropout 0.25 --batch_size 256 --run_name medium_largebatch

## Section 8 — Run Architecture 3: SmallResNet

In [ ]:
# Best config (with label smoothing — helps with noisy FER2013 labels)
!{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.3 --label_smoothing 0.1 --weight_decay 1e-4 --run_name resnet_best

In [ ]:
# No label smoothing — typically slightly more overfit
!{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.3 --label_smoothing 0.0 --run_name resnet_no_smoothing

In [ ]:
# High weight decay
!{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.2 --weight_decay 1e-3 --run_name resnet_high_wd

## Section 9 — Generate Wandb Report

After all runs finish:
1. Go to **wandb.ai** → your project **fer2013-emotion**
2. Click **Reports** → **New Report**
3. Add panels:
   - Line chart: `val_acc` grouped by `arch` (compare all 3 families)
   - Line chart: `train_acc` vs `val_acc` for each architecture (shows overfit/underfit)
   - Bar chart: `best_val_acc` per run
   - Media panel: confusion matrices
4. Add text sections explaining each architecture's decisions
5. Save and share the report link — paste it in your README


In [ ]:
# Optional: Print wandb project URL
print(f'Your wandb project: https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}')
print('Go there now to see all your logged runs and create a Report!')